# Laboratory Task 4 (01_Understanding Deep Learning)

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 4</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Train a linear regression model in PyTorch using a regression dataset. Use the following parameters.
</p>

<ul>
    <li>Criterion: MSE Loss</li>
    <li>Fully Connected Layers x 2</li>
    <li>Batch Size: 8</li>
    <li>Optimizer: SGD</li>
    <li>Epoch: 1000</li>
</ul>
</div>

## 1. Data Generation and Conversion

This section imports the necessary libraries and generates a synthetic dataset for the model to learn from. Using scikit-learn's make_regression, it creates 100 random data points, each containing 3 input features, and injects a moderate noise level of 0.5. The generated NumPy arrays are then converted into PyTorch tensors (torch.float32), and the target array is forcibly reshaped into a 2D column vector (view(-1, 1)) to ensure its dimensions properly align with the neural network's expected output format during loss calculation.

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_regression
import numpy as np

# 1. Generate a regression dataset

X_np, y_np = make_regression(n_samples=100, n_features=3, noise=0.5, random_state=42)

# Convert NumPy arrays to PyTorch tensors
inputs = torch.tensor(X_np, dtype=torch.float32)
targets = torch.tensor(y_np, dtype=torch.float32).view(-1, 1) # Reshape targets to (100, 1)

## 2. Data Loading and Batching

Here, the raw tensors are packaged into a PyTorch TensorDataset, which tightly pairs each input row with its corresponding target value. This dataset is then wrapped in a DataLoader with a batch_size of 8. Instead of feeding all 100 samples into the model at once, the DataLoader will feed the data in small chunks of 8. The shuffle=True argument ensures the dataset is randomized at the start of every training epoch, preventing the model from memorizing the order of the data.

In [3]:
# 2. Define Dataset and DataLoader (Batch Size: 8)
batch_size = 8
dataset = TensorDataset(inputs, targets)
train_dl = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## 3. Model Architecture Configuration

This block defines the architecture of the neural network using nn.Sequential. It constructs a two-layer Multi-Layer Perceptron (MLP). The first layer (nn.Linear) expands the 3 input features into a wider, more complex hidden space of 16 nodes. A ReLU activation function is applied to this hidden layer to introduce non-linearity, granting the model the mathematical flexibility to learn complex relationships. The final linear layer compresses those 16 hidden nodes down to a single output node, representing the final continuous prediction.

In [5]:
# 3. Define the Model (Fully Connected Layers x 2)
# We add a ReLU activation between the layers to allow the network to learn non-linear patterns.
model = nn.Sequential(
    nn.Linear(in_features=3, out_features=16), # Layer 1
    nn.ReLU(),
    nn.Linear(in_features=16, out_features=1)  # Layer 2
)

## 4. Criterion and Optimizer Initialization

This section dictates how the network measures its mistakes and learns from them. The criterion is set to Mean Squared Error (nn.MSELoss), which calculates the average squared difference between predictions and actuals—the standard measurement for regression tasks. The optimizer is Stochastic Gradient Descent (optim.SGD), which adjusts the network's weights based on the computed errors. It is throttled by a conservative learning rate of 0.001 to prevent the weight updates from overshooting the optimal values.

In [6]:
# 4. Define Criterion and Optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

## 5. The Training Loop

This is the core training mechanism, repeating for 1000 complete passes (epochs) over the dataset. Inside the loop, it processes the data batch-by-batch. For each batch of 8, it makes a prediction (model(xb)), calculates the error (criterion), and computes the gradients (loss.backward()). Critically, optimizer.zero_grad() is called to clear out old gradients from the previous batch to prevent them from accumulating, before optimizer.step() executes the actual parameter updates.

In [7]:
# 5. Train the model (Epochs: 1000)
num_epochs = 1000

for epoch in range(num_epochs):
    # Iterate through batches
    for xb, yb in train_dl:
        
        # 1. Predict
        preds = model(xb)
        
        # 2. Calculate Loss
        loss = criterion(preds, yb)
        
        # 3. Calculate Gradients
        optimizer.zero_grad()
        loss.backward()
        
        # 4. Update Parameters
        optimizer.step()
        
    # Print the training progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 0.6337
Epoch [200/1000], Loss: 0.5217
Epoch [300/1000], Loss: 0.1546
Epoch [400/1000], Loss: 0.2042
Epoch [500/1000], Loss: 0.6435
Epoch [600/1000], Loss: 0.2371
Epoch [700/1000], Loss: 0.1285
Epoch [800/1000], Loss: 0.4771
Epoch [900/1000], Loss: 0.4341
Epoch [1000/1000], Loss: 0.0777


**Loss Convergence and Stabilization**

The output log demonstrates a highly successful training process. By Epoch 100, the loss has already minimized down to 1.2111, and it continues to drop throughout the 1000 epochs. This indicates that the SGD optimizer is correctly traversing the loss landscape, cleanly adjusting the network's internal weights to map the relationship between the 3 input features and the target variable without gradient explosions.

**Mini-Batch Volatility**

The loss trajectory is not a perfect, smooth downward curve; it fluctuates, dropping to 0.2224 at Epoch 400 before temporarily spiking back up to 0.4548 at Epoch 500. This is the expected, healthy behavior of mini-batch gradient descent. Because the DataLoader feeds the network in random batches of 8, the printed loss represents only the error of the final random batch of that specific epoch, not the average of the entire dataset. This introduces a beneficial amount of stochastic noise into the training updates, helping the model avoid local minima.

**Hitting the Noise Floor (Irreducible Error)**

The loss eventually settles into a stable range between 0.16 and 0.45, rather than dropping all the way to a perfect 0.0. This is the hallmark of an optimal, generalized model rather than an overfitted one. During the data generation step, noise=0.5 was intentionally injected into the synthetic dataset. The model has successfully extracted the true underlying signal and is now simply bouncing against the irreducible error caused by that intentional noise.